In [1]:
import os
import subprocess
import sys
import time

def setup_kaggle_credentials():
    """Dynamically loads Kaggle credentials from secrets into environment variables.
    
    The Kaggle CLI automatically picks these up when executed in a subprocess,
    eliminating the need to write a physical 'kaggle.json' file to disk.
    """
    # 1. Try Kaggle Notebook Secrets (if running on Kaggle)
    try:
        from kaggle_secrets import UserSecretsClient
        user_secrets = UserSecretsClient()
        os.environ["KAGGLE_USERNAME"] = user_secrets.get_secret("KAGGLE_USERNAME")
        os.environ["KAGGLE_KEY"] = user_secrets.get_secret("KAGGLE_KEY")
        print("Successfully loaded credentials from Kaggle Secrets.")
        return
    except (ImportError, Exception):
        pass

    # 2. Try Google Colab Secrets (if running on Google Colab)
    try:
        from google.colab import userdata
        os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
        os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
        print("Successfully loaded credentials from Google Colab Secrets.")
        return
    except (ImportError, Exception):
        pass

    # 3. Fallback: Check if they are already in the system environment
    if "KAGGLE_USERNAME" in os.environ and "KAGGLE_KEY" in os.environ:
        print("Using pre-existing environment variables for Kaggle.")
        return

    print(
        "Warning: Kaggle credentials not found in Secrets/Environment.\n"
        "If you don't have a local ~/.kaggle/kaggle.json, the CLI commands will fail.",
        file=sys.stderr
    )

# Run the credential setup before executing CLI commands
setup_kaggle_credentials()

# Dynamically set KAGGLE_USER from the loaded secrets; fall back to hardcoded value if empty.
KAGGLE_USER = os.getenv("KAGGLE_USERNAME", "codingmaster24")

# Configuration dictionary
COMPETITIONS = {
    "titanic": {
        "comp_id": "titanic",
        "file": "pytorch_submission.csv",
        "notebook": "titanic-machine-learning-from-disaster-v1", 
        "version": "7",  # Note: double-check if v7 exists/has the submission file
        "message": "Titanic submission",
        "loop_count": 10  # Submit 20 times
    },
    "playground-series-s6e7": {
        "comp_id": "playground-series-s6e7",
        "file": "submission.csv",
        "notebook": "pytorch-health-condition-prediction-with-feature",
        "version": "5",
        "message": "PyTorch Health Condition Prediction with Feature",
        "loop_count": 100  # Submit 3 times
    },
    "neural-debris-removal-in-streak-detection-models": {
        "comp_id": "neural-debris-removal-in-streak-detection-models",
        "file": "submission.csv",
        "notebook": "neural-debris-removal-in-streak-detection-models",
        "version": "20",
        "message": "Neural Debris Removal in Streak Detection Models",
        "loop_count": 100  # Submit 5 times
    },
    "pokemon-tcg-ai-battle": {
        "comp_id": "pokemon-tcg-ai-battle",
        "file": "submission.tar.gz",
        "notebook": "ptcg-ai-battle-challenge-simulation",
        "version": "7",
        "message": "PTCG AI Battle Challenge Simulation",
        "loop_count": 5  # Submit 5 times
    },
    "kaggriculture": {
        "comp_id": "kaggriculture",
        "file": "submission.py",
        "notebook": "kaggriculture-homestead",
        "version": "2",
        "message": "Kaggriculture Homestead",
        "loop_count": 5  # Submit 5 times
    },
    "Predicting Smartphone Addiction": {
        "comp_id": "playground-series-s6e8",
        "file": "submission.csv",
        "notebook": "addiction-prediction-gbm",
        "version": "2",
        "message": "Predicting Smartphone Addiction",
        "loop_count": 10  # Submit 5 times
    },
    "AI Agent Security - Multi-Step Tool Attacks": {
        "comp_id": "ai-agent-security-multi-step-tool-attacks",
        "file": "submission.csv",
        "notebook": "ai-agent-security",
        "version": "3",
        "message": "AI Agent Security",
        "loop_count": 5  # Submit 5 times
    },
    "ulsu-titanic-comp": {
        "comp_id": "ulsu-titanic-comp",
        "file": "submission.csv",
        "notebook": "titanic-machine-learning-from-disaster",
        "version": "3",
        "message": "Titanic Machine Learning from Disaster",
        "loop_count": 10  # Submit 5 times
    },
    "arc_prize": {
        "comp_id": "arc-prize-2026-arc-agi-3",
        "file": "submission.parquet",
        "notebook": "arc-agi-3-v1",
        "version": "2",
        "message": "ARC Prize 2026 submission",
        "loop_count": 2  # Submit 3 times
    }
}

def execute_submission(config, iteration):
    """Constructs and executes a single Kaggle CLI command without a popup window."""
    # Appending iteration number to message to ensure unique submission names
    msg = f"{config['message']} (submission {iteration})"
    
    command = [
        "kaggle", "competitions", "submit",
        "-c", config["comp_id"],
        "-f", config["file"],
        "-k", f"{KAGGLE_USER}/{config['notebook']}",
        "-v", str(config["version"]),
        "-m", msg
    ]
    
    print(f"Executing: {' '.join(command)}")
    
    # Hide the console window popup on Windows
    creation_flags = subprocess.CREATE_NO_WINDOW if sys.platform == "win32" else 0
    
    try:
        result = subprocess.run(
            command, 
            check=True, 
            text=True, 
            capture_output=True,
            creationflags=creation_flags
        )
        print(f"-> Success (Loop {iteration})")
        return True
    except subprocess.CalledProcessError as e:
        print(f"-> Failed (Loop {iteration})")
        print(e.stderr.strip(), file=sys.stderr)
        return False

def submit_all_with_loops():
    """Iterates through competitions and runs them their respective number of times."""
    print("Starting automated loop submissions...\n" + "="*50)
    
    for key, config in COMPETITIONS.items():
        print(f"\nProcessing {key.upper()} (Target: {config['loop_count']} submissions)")
        print("-" * 50)
        
        for i in range(1, config["loop_count"] + 1):
            execute_submission(config, i)
            
            # Short pause to prevent API rate limiting issues
            time.sleep(10) 
            
        print("="*50)

if __name__ == "__main__":
    submit_all_with_loops()